# 05 SHAP-蜂窝图解释（任务书：SHAP、分区主控因子变化）

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline

# ---- bootstrap: src ----
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, ensure_dir, safe_col
from src.holdout import build_preprocessor, build_model
from src.shap_plots import shap_importance_combo

cfg = load_config(ROOT / "config" / "config.yaml")
OUT = ensure_dir(ROOT / "outputs")
SHAP_DIR = ensure_dir(OUT / "shap")

# ---- data ----
p_model = OUT / "clean_data.parquet"
df = pd.read_parquet(p_model)

bcf_col = safe_col(df, cfg["data"]["columns"]["target_bcf"])
seed = int(cfg.get("project", {}).get("seed", 42))

# speed controls
sample_rows = int(cfg.get("shap", {}).get("sample_rows", 800))
top_k = int(cfg.get("shap", {}).get("top_k", 15))
max_display = int(cfg.get("shap", {}).get("max_display", 20))


def _safe_key(x):
    s = str(x)
    return s.replace(" ", "").replace("(", "").replace(")", "").replace(",", "_").replace("'", "")


def _to_dense(X):
    try:
        import scipy.sparse as sp
        if sp.issparse(X):
            return X.toarray()
    except Exception:
        pass
    return np.asarray(X)


def _compute_shap_values(est, Xt, model_name, random_seed=42):
    mname = str(model_name).lower()
    tree_models = {"xgb", "rf", "cat", "catboost", "lgbm", "lightgbm"}
    if mname in tree_models:
        explainer = shap.TreeExplainer(est)
        sv = explainer.shap_values(Xt)
        if isinstance(sv, list):
            sv = sv[0]
        return np.asarray(sv, dtype=float)

    Xt_np = np.asarray(Xt, dtype=float)
    rng = np.random.default_rng(random_seed)
    n_bg = min(120, Xt_np.shape[0])
    bg_idx = rng.choice(Xt_np.shape[0], size=n_bg, replace=False)
    background = Xt_np[bg_idx]
    explainer = shap.Explainer(est.predict, background)
    sv = explainer(Xt_np).values
    return np.asarray(sv, dtype=float)


# ------------------------------------------------------------------
# 1) Select ONE global best model from existing 7:3 holdout results
# ------------------------------------------------------------------
hcfg = cfg.get("holdout", {}) or {}
out_subdir = str(hcfg.get("out_subdir", "holdout_70_30"))
metrics_path = OUT / out_subdir / "metrics_holdout.xlsx"
if not metrics_path.exists():
    raise RuntimeError(f"missing holdout metrics: {metrics_path}")

metrics = pd.read_excel(metrics_path)
allow = {"xgb", "rf", "svm", "ann", "cat", "catboost", "lgbm", "lightgbm"}
candidates = [str(m).lower() for m in cfg.get("modeling", {}).get("models", []) if str(m).lower() in allow]
if not candidates:
    raise RuntimeError("config.yaml ? modeling.models ????? SHAP ???")

m0 = metrics.copy()
m0["model"] = m0["model"].astype(str).str.lower()
m0 = m0[m0["model"].isin(candidates)].copy()
if m0.empty:
    raise RuntimeError("holdout ?????? modeling.models ?????")

board = (
    m0.groupby("model", as_index=False)
    .apply(lambda g: pd.Series({
        "r2_weighted": np.average(g["r2"], weights=g["n_test"]),
        "rmse_weighted": np.average(g["rmse"], weights=g["n_test"]),
        "mae_weighted": np.average(g["mae"], weights=g["n_test"]),
        "groups_used": int(g["group"].nunique()),
    }))
    .reset_index(drop=True)
    .sort_values(["r2_weighted", "rmse_weighted"], ascending=[False, True])
)

m = str(board.iloc[0]["model"])
print("Global model board from 7:3 holdout metrics:")
print(board.to_string(index=False))
print()
print(f"Selected ONE global model for SHAP: {m}")


# ------------------------------------------------------------------
# 1.5) Model comparison bar charts (global + by-group)
# ------------------------------------------------------------------
CMP_DIR = ensure_dir(SHAP_DIR / "model_compare")

# project-aligned palette (earth + slate)
PALETTE = ["#80B5BA", "#CCE7E3", "#EDC4A7", "#E4A8AA", "#B7A8C9", "#9FB6D9", "#BFD7C1", "#D6C5B2"]

def _model_color_map(models):
    ms = [str(x) for x in models]
    return {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(ms)}

# Save model board table
board.to_excel(CMP_DIR / "model_board_global_weighted.xlsx", index=False)

# ---- global weighted comparison (R2 / RMSE / MAE) ----
b = board.copy()
model_order = b["model"].tolist()
cm = _model_color_map(model_order)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), dpi=220)
metrics_global = [
    ("r2_weighted", "Weighted R2", False),
    ("rmse_weighted", "Weighted RMSE", True),
    ("mae_weighted", "Weighted MAE", True),
]
for ax, (col, title, asc) in zip(axes, metrics_global):
    d = b.sort_values(col, ascending=asc)
    ax.bar(d["model"], d[col], color=[cm[m] for m in d["model"]], edgecolor="#1F2937", linewidth=0.7)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Model")
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Global Model Comparison (from 7:3 Holdout)", fontweight="bold", y=1.03)
fig.tight_layout()
fig.savefig(CMP_DIR / "global_model_compare_r2_rmse_mae.png", bbox_inches="tight")
plt.close(fig)

# ---- by-group comparison for each metric (group x model bars) ----
mplot = m0.copy()
for metric, title, asc in [("r2", "R2", False), ("rmse", "RMSE", True), ("mae", "MAE", True)]:
    piv = mplot.pivot_table(index="group", columns="model", values=metric, aggfunc="mean")
    cols = [c for c in model_order if c in piv.columns] + [c for c in piv.columns if c not in model_order]
    piv = piv[cols]

    n_groups = len(piv.index)
    x = np.arange(n_groups)
    width = 0.78 / max(1, len(piv.columns))

    fig, ax = plt.subplots(figsize=(max(12, n_groups * 0.9), 5.2), dpi=220)
    for i, col in enumerate(piv.columns):
        ax.bar(x + (i - (len(piv.columns) - 1) / 2) * width,
               piv[col].values,
               width=width,
               label=col,
               color=cm.get(col, PALETTE[i % len(PALETTE)]),
               edgecolor="#1F2937",
               linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels([str(g) for g in piv.index], rotation=28, ha="right")
    ax.set_title(f"By-Group Model Comparison: {title}", fontweight="bold")
    ax.set_ylabel(title)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    ax.legend(ncol=min(6, len(piv.columns)), frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(CMP_DIR / f"by_group_{metric}_compare.png", bbox_inches="tight")
    plt.close(fig)

# ---- one chart per group: compare all models on R2/RMSE/MAE ----
for g, gdf in mplot.groupby("group"):
    gdf = gdf.copy().sort_values(["r2", "rmse"], ascending=[False, True])
    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2), dpi=220)
    for ax, metric, title, asc in [
        (axes[0], "r2", "R2", False),
        (axes[1], "rmse", "RMSE", True),
        (axes[2], "mae", "MAE", True),
    ]:
        d = gdf.sort_values(metric, ascending=asc)
        ax.bar(d["model"], d[metric], color=[cm.get(m, "#2F6C74") for m in d["model"]], edgecolor="#1F2937", linewidth=0.6)
        ax.set_title(title, fontweight="bold")
        ax.grid(axis="y", alpha=0.25, linestyle="--")
        ax.tick_params(axis="x", rotation=22)
    fig.suptitle(f"Model Comparison | {g}", fontweight="bold", y=1.04)
    fig.tight_layout()
    gsafe = _safe_key(g)
    fig.savefig(CMP_DIR / f"group_model_compare_{gsafe}.png", bbox_inches="tight")
    plt.close(fig)

print("saved compare charts to:", CMP_DIR)

# ------------------------------------------------------------------
# 2) Use selected model for ALL groups (no spatial CV)
# ------------------------------------------------------------------
group_keys = ["crop", "ph_bin"] if "crop" in df.columns else ["ph_bin"]
min_rows = int((cfg.get("holdout", {}) or {}).get("min_rows_per_group", 20))

# Raw-column exclusions: remove group/source/id-like columns from SHAP modeling.
def_exclude_raw = {
    "source_sheet", "sheet_name", "ph_bin_from_sheet", "lod_used", "bcf_calc",
    "crop", "ph_bin", "group", "group_id", "sample_id", "id", "index"
}
user_exclude_raw = set(cfg.get("shap", {}).get("exclude_raw_cols", []))
exclude_raw_cols = def_exclude_raw | user_exclude_raw

# Encoded-feature exclusions: second guard for one-hot names and metadata tokens.
def _drop_encoded_feature(name: str) -> bool:
    n = str(name).lower()
    bad_prefix = ("crop_", "ph_bin_", "source_sheet", "group_")
    bad_tokens = ("_id", "sample_id", "source_sheet", "ph_bin_from_sheet", "lod_used", "bcf_calc")
    return n.startswith(bad_prefix) or any(tok in n for tok in bad_tokens)

# base drop list from config + our raw exclusions
drop_from_model = set((cfg.get("data", {}) or {}).get("drop_from_model", [])) | exclude_raw_cols

summary_rows = []
for key, sub in df.groupby(group_keys, dropna=False):
    sub = sub.reset_index(drop=True)

    y = pd.to_numeric(sub[bcf_col], errors="coerce")
    mask = y.notna()
    sub = sub.loc[mask].reset_index(drop=True)
    y = y.loc[mask].astype(float).reset_index(drop=True)

    if len(sub) < min_rows:
        continue

    X = sub.drop(columns=[bcf_col], errors="ignore").copy()
    X = X.drop(columns=[c for c in drop_from_model if c in X.columns], errors="ignore")
    X = X.dropna(axis=1, how="all")

    m2 = np.isfinite(y.to_numpy())
    X = X.loc[m2].reset_index(drop=True)
    y = y.loc[m2].reset_index(drop=True)
    if len(X) < min_rows:
        continue

    base = build_model(m, seed=seed)
    if base is None:
        raise RuntimeError(f"model '{m}' is unavailable in current env")

    pre = build_preprocessor(X)
    pipe = Pipeline(steps=[("pre", pre), ("model", base)])
    pipe.fit(X, y)

    if len(X) > sample_rows:
        X_used = X.sample(sample_rows, random_state=seed).reset_index(drop=True)
    else:
        X_used = X.reset_index(drop=True)

    Xt = _to_dense(pre.transform(X_used))

    try:
        feat_names = list(pre.get_feature_names_out())
    except Exception:
        feat_names = [f"f{i}" for i in range(Xt.shape[1])]

    # second guard: remove unwanted encoded columns in SHAP output/plot
    keep_idx = [i for i, fn in enumerate(feat_names) if not _drop_encoded_feature(fn)]
    if not keep_idx:
        print(f"[skip] {key}: all encoded features filtered")
        continue
    Xt = Xt[:, keep_idx]
    feat_names = [feat_names[i] for i in keep_idx]

    est = pipe.named_steps["model"]
    sv_all = _compute_shap_values(est, _to_dense(pre.transform(X_used)), m, random_seed=seed)
    sv = sv_all[:, keep_idx]

    if sv.shape[0] != Xt.shape[0]:
        sv = sv[: Xt.shape[0], :]
    if sv.shape[1] != Xt.shape[1]:
        sv = sv[:, : Xt.shape[1]]

    key_str = "_".join(_safe_key(k) for k in (key if isinstance(key, tuple) else (key,)))
    out_png = SHAP_DIR / f"shap_combo_{key_str}_{m}.png"

    shap_importance_combo(
        shap_values=sv,
        X=Xt,
        feature_names=feat_names,
        out_png=out_png,
        title=f"{key} | {m} | n={sv.shape[0]}",
        max_display=max_display,
    )
    print("saved:", out_png)

    mean_abs = np.mean(np.abs(sv), axis=0)
    idx = np.argsort(mean_abs)[::-1][:top_k]
    for i in idx:
        summary_rows.append({
            "group": str(key),
            "model_for_shap": m,
            "feature": feat_names[i] if i < len(feat_names) else f"f{i}",
            "mean_abs_shap": float(mean_abs[i]),
            "model_select_source": f"{out_subdir}/metrics_holdout.xlsx",
        })

rank = pd.DataFrame(summary_rows)
rank.to_excel(OUT / "shap_rank_compare.xlsx", index=False)
rank.head(20)





